# ROC-Kurven einfach erklärt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_12/roc_kurve_demo.ipynb)

**Lernziel:** Verstehen, wie ROC-Kurven entstehen und interpretiert werden.

**Beispiel:** Defekterkennung in der Produktion - 100 Bauteile werden getestet.

Wir trainieren hier kein Modell, sondern erzeugen Beispieldaten. So können wir uns ganz auf die ROC-Kurve konzentrieren. Bitte führe die Zellen von oben nach unten aus.

**Herkunft und Bedeutung**

Der Begriff stammt nicht ursprünglich aus der Statistik oder dem maschinellen Lernen, sondern aus der Radar- und Signalverarbeitung während des Zweiten Weltkriegs:

„Receiver“ = Empfänger (Radar-Empfangsgerät)

„Operating Characteristic“ = beschreibt, wie dieser Empfänger unter verschiedenen Schwellwert-Einstellungen arbeitet

Damals wollte man herausfinden, wie gut ein Radar zwischen „Signal vorhanden“ (z. B. Flugzeug erkannt) und „kein Signal“ (Rauschen) unterscheiden kann.
Dazu variierte man den Entscheidungsschwellenwert: je niedriger er war, desto mehr Treffer, aber auch mehr Fehlalarme; je höher, desto weniger Fehlalarme, aber auch mehr verpasste Signale.
Die ROC-Kurve zeigte diesen Trade-off zwischen Trefferquote (True Positive Rate) und Falschalarmrate (False Positive Rate).

**Übertragung auf maschinelles Lernen**

In der Statistik und im Machine Learning nutzt man die ROC-Kurve heute genau für denselben Zweck:

Sie zeigt, wie gut ein Klassifikationsmodell bei verschiedenen Schwellwerten zwischen zwei Klassen unterscheiden kann.

Die Achsen sind:

x-Achse: False Positive Rate (1 − Spezifität)

y-Achse: True Positive Rate (Sensitivität)

Der Flächeninhalt unter der ROC-Kurve (AUC) dient als Maß für die Trennschärfe des Modells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("✓ Bibliotheken geladen")

## 📊 Unser Beispiel-Datensatz

**100 Bauteile getestet:**
- 90 sind tatsächlich **OK** (negative Klasse, Label 0)
- 10 sind tatsächlich **NOK/defekt** (positive Klasse, Label 1)

**Für jedes Teil erzeugen wir einen Defektwert zwischen 0 und 1:**
- Hoher Wert → stärkerer Verdacht auf einen Defekt
- Niedriger Wert → geringerer Verdacht auf einen Defekt

Stell dir den Wert wie eine Warnanzeige vor: Je höher sie ausschlägt, desto eher prüfen wir das Teil. Ein Wert von 0,7 bedeutet hier **nicht automatisch 70 % Defektwahrscheinlichkeit**. Unsere Werte sind nur für das Beispiel erzeugt.

Bei einem echten Modell müsste man erst prüfen, ob seine Wahrscheinlichkeitsangaben zuverlässig sind. Für die ROC-Kurve reicht es, die Teile nach ihrem Defektwert zu ordnen.

**Die finale Entscheidung "defekt oder nicht" wird mit einem Schwellwert getroffen:**

Zum Beispiel bei Schwellwert 0,5: Defektwert ≥ 0,5 → als defekt einstufen. Das ist die Entscheidung des Systems, noch kein Beweis für einen tatsächlichen Defekt.

In [ ]:
# Einfaches Beispiel mit Fehlklassifikationen
np.random.seed(42)  # Für reproduzierbare Ergebnisse

# Wahre Labels: 90x OK (0), 10x defekt (1)
wahre_labels = [0] * 90 + [1] * 10

# Zufallswerte erzeugen und für das Beispiel auf 0.05 bis 0.95 begrenzen
# OK-Teile: Normalverteilung um 0.3 (manche haben trotzdem höhere Werte!)
ok_defektwerte = np.random.normal(0.3, 0.15, 90)
ok_defektwerte = np.clip(ok_defektwerte, 0.05, 0.95)

# Defekte Teile: Normalverteilung um 0.7 (manche haben trotzdem niedrigere Werte!)
nok_defektwerte = np.random.normal(0.7, 0.15, 10)
nok_defektwerte = np.clip(nok_defektwerte, 0.05, 0.95)

defektwerte = np.concatenate([ok_defektwerte, nok_defektwerte])

print(f"📊 BEISPIELDATENSATZ:")
print(f"Total: {len(wahre_labels)} Bauteile")
print(f"OK-Teile: {wahre_labels.count(0)}")
print(f"Defekte Teile: {wahre_labels.count(1)}")
print(f"\nBeispiel-Defektwerte:")
print(f"OK-Teile: {ok_defektwerte[:5].round(3)} ...")
print(f"Defekte Teile: {nok_defektwerte.round(3)}")

print(f"\n🔍 WICHTIG: Überlappungen vorhanden!")
print(f"Höchster Defektwert eines OK-Teils: {ok_defektwerte.max():.3f}")
print(f"Niedrigster Defektwert eines defekten Teils: {nok_defektwerte.min():.3f}")
print("→ Prüfe: Liegt ein OK-Teil über einem defekten Teil? Dann trennt keine einzelne Schwelle alle Teile richtig.")

# Zeige problematische Fälle
hohe_ok = ok_defektwerte[ok_defektwerte >= 0.5]
niedrige_defekt = nok_defektwerte[nok_defektwerte < 0.5]

print(f"\n⚠️ PROBLEMATISCHE FÄLLE:")
print(f"OK-Teile mit Defektwert >= 0.5 (Fehlalarm, FP): {len(hohe_ok)} Stück")
print(f"Defekte Teile mit Defektwert < 0.5 (übersehener Defekt, FN): {len(niedrige_defekt)} Stück")
print(f"→ Diese werden bei Schwellwert 0.5 falsch klassifiziert!")

## 🎯 Was ist der Schwellwert?

**Der Schwellwert entscheidet die Klassifikation:**
- Defektwert ≥ Schwellwert → "Defekt" vorhersagen
- Defektwert < Schwellwert → "OK" vorhersagen

**Problem:** Die Defektwerte überlappen sich!
- Manche OK-Teile haben hohe Werte (Fehlalarm)
- Manche defekte Teile haben niedrige Werte (übersehen)

**Beispiel bei Schwellwert 0.5:**
- OK-Teil mit 0.6 → fälschlich als "Defekt" klassifiziert (FP)
- Defektes Teil mit 0.4 → fälschlich als "OK" klassifiziert (FN)

In [ ]:
def berechne_metriken(wahre_labels, defektwerte, schwellwert):
    """Berechnet TPR und FPR für einen gegebenen Schwellwert"""
    if len(wahre_labels) != len(defektwerte):
        raise ValueError('Für jedes Teil werden ein Label und ein Defektwert benötigt.')
    if set(wahre_labels) != {0, 1}:
        raise ValueError('Für diese ROC-Berechnung werden sowohl OK- als auch NOK-Teile benötigt (Labels 0 und 1).')
    
    # Vorhersagen basierend auf Schwellwert
    vorhersagen = [1 if wert >= schwellwert else 0 for wert in defektwerte]
    
    # Konfusionsmatrix berechnen
    TP = sum(1 for true, pred in zip(wahre_labels, vorhersagen) if true == 1 and pred == 1)
    TN = sum(1 for true, pred in zip(wahre_labels, vorhersagen) if true == 0 and pred == 0)
    FP = sum(1 for true, pred in zip(wahre_labels, vorhersagen) if true == 0 and pred == 1)
    FN = sum(1 for true, pred in zip(wahre_labels, vorhersagen) if true == 1 and pred == 0)
    
    # TPR und FPR berechnen
    TPR = TP / (TP + FN)
    FPR = FP / (FP + TN)
    
    return TPR, FPR, TP, TN, FP, FN

# Beispiel für Schwellwert 0.5
schwellwert_beispiel = 0.5
tpr, fpr, tp, tn, fp, fn = berechne_metriken(wahre_labels, defektwerte, schwellwert_beispiel)

print(f"📐 BEISPIEL: Schwellwert = {schwellwert_beispiel}")
print(f"="*40)
print(f"TP (richtig als defekt erkannt): {tp}")
print(f"TN (richtig als OK erkannt): {tn}")
print(f"FP (OK fälschlich als defekt): {fp}")
print(f"FN (defekt fälschlich als OK): {fn}")
print(f"")
print(f"TPR = {tp}/({tp}+{fn}) = {tpr:.3f} = {tpr*100:.1f}%")
print(f"FPR = {fp}/({fp}+{tn}) = {fpr:.3f} = {fpr*100:.1f}%")
print(f"")
print(f"Interpretation:")
print(f"• {tpr*100:.1f}% aller defekten Teile werden erkannt")
print(f"• {fpr*100:.1f}% aller OK-Teile werden fälschlich als defekt eingestuft")

## 📈 ROC-Kurve: Verschiedene Schwellwerte testen

**Idee der ROC-Kurve:**
1. Teste viele verschiedene Schwellwerte (0.1, 0.2, 0.3, ..., 0.9)
2. Berechne für jeden Schwellwert TPR und FPR
3. Trage jeden Punkt (FPR, TPR) in ein Diagramm ein
4. Verbinde die Punkte → ROC-Kurve!

Die Tabelle zeigt zunächst nur neun leicht lesbare Beispiele. Für die **vollständige Kurve** müssen wir jeden unterschiedlichen Defektwert als Schwelle verwenden. Nur dort ändert sich die Entscheidung für mindestens ein Teil.

Wir beginnen außerdem mit einer Schwelle oberhalb aller Werte: Dann wird kein Teil als defekt eingestuft, also starten wir bei (0, 0). Bei der niedrigsten Schwelle werden alle Teile als defekt eingestuft: Das ergibt (1, 1).

In [ ]:
# Verschiedene Schwellwerte testen
schwellwerte = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
tpr_werte = []
fpr_werte = []

print("📈 ROC-KURVE SCHRITT FÜR SCHRITT:")
print("="*50)
print("Schwellwert | TPR   | FPR   | Übersehen | Fehlalarme")
print("-"*50)

for schwellwert in schwellwerte:
    tpr, fpr, tp, tn, fp, fn = berechne_metriken(wahre_labels, defektwerte, schwellwert)
    
    tpr_werte.append(tpr)
    fpr_werte.append(fpr)
    
    print(f"   {schwellwert:.1f}      | {tpr:.3f} | {fpr:.3f} | {fn:9d} | {fp:10d}")

print(f"\n🎯 BEOBACHTUNG:")
print("• Senken wir die Schwelle, werden mehr oder gleich viele Teile als defekt eingestuft.")
print("• Dadurch übersehen wir weniger oder gleich viele Defekte, haben aber mehr oder gleich viele Fehlalarme.")
print("• Wir müssen also zwischen übersehenen Defekten und Fehlalarmen abwägen.")

In [ ]:
# 1.1 liegt über allen Defektwerten zwischen 0 und 1: Startpunkt (0, 0).
roc_schwellwerte = [1.1] + sorted(set(defektwerte), reverse=True)
roc_tpr = []
roc_fpr = []
for schwellwert in roc_schwellwerte:
    tpr, fpr, tp, tn, fp, fn = berechne_metriken(wahre_labels, defektwerte, schwellwert)
    roc_tpr.append(tpr)
    roc_fpr.append(fpr)

plt.figure(figsize=(12, 5))

# Subplot 1: ROC-Kurve
plt.subplot(1, 2, 1)
plt.plot(roc_fpr, roc_tpr, 'b-', linewidth=2, label='Vollständige ROC-Kurve')
plt.plot(fpr_werte, tpr_werte, 'bo', markersize=5, label='Beispiele aus der Tabelle')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, alpha=0.7, label='Zufall')

# Einige Schwellwerte beschriften
for i, (fpr, tpr, schwellwert) in enumerate(zip(fpr_werte, tpr_werte, schwellwerte)):
    if i % 3 == 0:  # Nur jeden dritten Punkt beschriften
        plt.annotate(f'{schwellwert:.1f}', (fpr, tpr), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('FPR (False Positive Rate)')
plt.ylabel('TPR (True Positive Rate)')
plt.title('ROC-Kurve')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: TPR und TNR vs Schwellwert
plt.subplot(1, 2, 2)
plt.plot(schwellwerte, tpr_werte, 'go-', linewidth=2, label='TPR (Sensitivität)')
plt.plot(schwellwerte, [1-fpr for fpr in fpr_werte], 'ro-', linewidth=2, label='TNR (Spezifität)')
plt.xlabel('Schwellwert')
plt.ylabel('Rate')
plt.title('TPR und TNR vs Schwellwert')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# AUC berechnen
auc_wert = np.trapezoid(roc_tpr, roc_fpr)
print(f"🎯 ROC-KURVE ERSTELLT:")
print(f"• AUC = {auc_wert:.3f} (Fläche unter der vollständigen Kurve)")
print("• Eine AUC nahe 1 bedeutet: Defekte Teile erhalten meist höhere Werte als OK-Teile.")
print("• Eine AUC von 0.5 entspricht beim Ordnen der Teile dem Zufallsniveau.")
print("• Die AUC sagt nicht, welche Schwelle für unseren Betrieb am besten ist.")

In [ ]:
# Für beide Gruppen dieselben Balkengrenzen verwenden
plt.figure(figsize=(10, 6))

balkengrenzen = np.linspace(0, 1, 21)
plt.hist(ok_defektwerte, bins=balkengrenzen, alpha=0.7, label='OK-Teile (90 Stück)', color='green')
plt.hist(nok_defektwerte, bins=balkengrenzen, alpha=0.7, label='Defekte Teile (10 Stück)', color='red')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Schwellwert 0.5')

# Fehlklassifikations-Bereiche markieren
plt.axvspan(0.5, 1.0, alpha=0.2, color='red')
plt.axvspan(0.0, 0.5, alpha=0.2, color='orange')

plt.xlabel('Defektwert (höher = stärkerer Verdacht)')
plt.ylabel('Anzahl Bauteile')
plt.title('Defektwerte von OK- und NOK-Teilen')
plt.legend()

# Erklärende Texte
plt.text(0.25, plt.ylim()[1]*0.7, 'FN-Bereich:\nDefekte als OK', ha='center', fontsize=10,
         bbox=dict(boxstyle="round", facecolor="orange", alpha=0.8))
plt.text(0.75, plt.ylim()[1]*0.7, 'FP-Bereich:\nOK als defekt', ha='center', fontsize=10,
         bbox=dict(boxstyle="round", facecolor="red", alpha=0.8))

plt.tight_layout()
plt.show()

print(f"🔍 ÜBERLAPPUNGS-PROBLEM:")
print(f"• Grüne Balken rechts von 0.5 = Fehlalarme (FP)")
print(f"• Rote Balken links von 0.5 = Übersehene Defekte (FN)")
print("• In diesem Datensatz trennt keine einzelne Schwelle alle Teile richtig.")
print("• Die Balken zeigen Anzahlen: Es gibt hier deutlich mehr OK- als NOK-Teile.")

## 🎯 Optimalen Schwellwert wählen

**Die Frage:** Welcher Punkt auf der ROC-Kurve ist der beste?

**Die Antwort:** Hängt vom Anwendungsfall ab!

### 📝 Was bedeuten die Strategien?

**🚨 Vorsichtig (niedriger Schwellwert):**
- System sagt schnell "Das ist defekt!" 
- Schon bei einem niedrigen Defektwert wird "defekt" gemeldet
- ➤ Wenig wird übersehen, aber viele Fehlalarme

**⚖️ Vergleichswert (Schwellwert 0,5):**
- Ab einem Defektwert von 0,5 wird "defekt" gemeldet
- Das ist ein einfacher Startwert, nicht automatisch die beste Lösung

**🛡️ Zurückhaltend (hoher Schwellwert):**
- System sagt erst bei einem hohen Defektwert "Das ist defekt!"
- Es muss also mehr Verdacht vorliegen, bevor ein Alarm ausgelöst wird
- ➤ Wenige Fehlalarme, aber mehr wird übersehen

### 🎯 Anwendungsbeispiele:
- **Sicherheitskritisches Bauteil:** Ein übersehener Defekt kann gefährlich sein. Dann nehmen wir eher zusätzliche Prüfungen durch Fehlalarme in Kauf.
- **Spam-Filter:** Eine wichtige Nachricht im Spam-Ordner kann ärgerlicher sein als eine zusätzliche Werbemail im Posteingang. Dann wählen wir eher eine höhere Schwelle.

Entscheidend sind die Folgen beider Fehlerarten und wie häufig Defekte vorkommen. Eine Schwelle von 0,5 ist nicht allein deshalb gut, weil sie in der Mitte liegt.

In [ ]:
# Drei verschiedene Strategien vergleichen
strategien = {
    'Vorsichtig (0.3)': 0.3,  # Niedrig = sagt schnell "defekt!"
    'Vergleichswert (0.5)': 0.5,
    'Zurückhaltend (0.7)': 0.7   # Hoch = Alarm erst bei hohem Defektwert
}

print("🎯 SCHWELLWERT-STRATEGIEN VERGLEICH:")
print("="*60)

for name, schwellwert in strategien.items():
    tpr, fpr, tp, tn, fp, fn = berechne_metriken(wahre_labels, defektwerte, schwellwert)
    
    print(f"\n📊 {name.upper()}:")
    print(f"   Schwellwert: {schwellwert}")
    print(f"   TPR: {tpr:.1%} → {tp} von {tp+fn} defekten Teilen erkannt")
    print(f"   FPR: {fpr:.1%} → {fp} von {fp+tn} OK-Teilen als Fehlalarm")
    print(f"   Übersehene defekte Teile: {fn}")
    print(f"   Fälschlich beanstandete OK-Teile: {fp}")
    
    # Bewertung mit Erklärung
    if schwellwert == 0.3:
        print(f"   ➤ VORSICHTIG: Sagt schnell 'defekt!' → wenig übersehen, viele Fehlalarme")
    elif schwellwert == 0.5:
        print("   ➤ VERGLEICHSWERT: 0.5 ist ein Startwert, nicht automatisch die beste Schwelle.")
    else:
        print("   ➤ ZURÜCKHALTEND: Alarm erst bei hohem Defektwert; dabei können mehr Defekte übersehen werden.")

# Visualisierung der Strategien
plt.figure(figsize=(10, 6))

# ROC-Kurve mit markierten Strategien
plt.plot(roc_fpr, roc_tpr, 'b-', linewidth=2, alpha=0.7, label='Vollständige ROC-Kurve')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Zufall')

# Strategien als Punkte markieren
farben = ['red', 'orange', 'green']
for i, (name, schwellwert) in enumerate(strategien.items()):
    # Finde den entsprechenden Punkt
    idx = schwellwerte.index(schwellwert)
    plt.plot(fpr_werte[idx], tpr_werte[idx], 'o', 
            color=farben[i], markersize=12, label=name)
    
    # Beschriftung
    plt.annotate(name.split('(')[0], 
                (fpr_werte[idx], tpr_werte[idx]),
                xytext=(10, 10), textcoords='offset points', 
                fontsize=10, fontweight='bold')

plt.xlabel('FPR (False Positive Rate)')
plt.ylabel('TPR (True Positive Rate)')
plt.title('ROC-Kurve mit verschiedenen Strategien')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n💡 FAZIT:")
print("• Die ROC-Kurve zeigt, wie sich Treffer und Fehlalarme mit der Schwelle ändern.")
print("• Die passende Schwelle hängt von den Folgen beider Fehlerarten ab.")
print("• Auch übersehene Defekte können teuer oder gefährlich sein.")
print("• Bei einem echten Modell prüfen wir die Wahl zusätzlich an neuen, noch nicht zum Lernen verwendeten Daten.")

## 📚 Zusammenfassung: ROC-Kurven verstehen

**Was wir gelernt haben:**

1. **Schwellwerte** entscheiden, ab welchem Defektwert wir "positiv", also "defekt", vorhersagen

2. **ROC-Kurve entsteht** durch Variation des Schwellwerts:
   - Jeder Schwellwert → ein Punkt (FPR, TPR)
   - Alle Punkte verbinden → ROC-Kurve

3. **Interpretation:**
   - Links oben = gut (hohe TPR, niedrige FPR)
   - Diagonale = Zufall (nutzlos)
   - AUC = Fläche unter der vollständigen Kurve: Wie gut stehen defekte Teile in der Reihenfolge vor OK-Teilen?
   - Ein hoher Defektwert ist nicht automatisch eine verlässliche Prozentangabe

4. **Optimaler Schwellwert** hängt vom Anwendungsfall ab:
   - Was ist schlimmer: einen Defekt übersehen oder ein OK-Teil unnötig prüfen?
   - Die Mitte bei 0,5 ist nicht automatisch der beste Kompromiss